In [1]:
import os
import pandas as pd
import numpy as np
import joblib
from pmdarima import ARIMA, auto_arima
from src.common.utils import get_root_directory, collate_array_elements, make_directory
from src.common.stats import root_mean_squared_percentage_error
import numpy as np
from src.preprocessing.data_loader import DataLoader
from sklearn.metrics import root_mean_squared_error, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, precision_score, accuracy_score
import mlflow

In [2]:
# PARAMETERS TO CHANGE
split_size = 0.15
start_date = "01/01/2021"
end_date = "31/12/2023"
maxiter=500
n_periods=7
model_type = "ARIMA"
split_name = "validation"
experiment_name = f"{model_type}_train_{split_name}"
max_p = 7
max_q = 1
d = 1
test_p = 1
test_q = 1

In [3]:
root_dir = get_root_directory()
DL = DataLoader(root_dir)
DL.load_data()
DL.set_time_range(start_date=start_date, end_date=end_date)
if split_name=="validation":
    train, test = DL.split_data(split_size=split_size)
    train, val = DL.split_data(split_type="train_val",split_size=split_size)
    test = test["ETH_D_AvgPrc"]
    train = train["ETH_D_AvgPrc"]
    val = val["ETH_D_AvgPrc"]
    split = val[:-n_periods]
elif split_name=="test":
    train, test = DL.split_data(split_size=0.15)
    test = test["ETH_D_AvgPrc"]
    train = train["ETH_D_AvgPrc"]
    split = test[:-n_periods]

In [5]:
mlflow.set_tracking_uri(f"sqlite:///{root_dir}/mlruns/mlruns.db")
mlflow.set_experiment(experiment_name)

for p in range(1,max_p+1):
    for q in range(1,max_q+1):
        predictions = []
        train_recursive = list(train)
        order = (p, d, q)
        run_name = f"order_({p},{d},{q})"
        with mlflow.start_run(run_name=run_name):
            model = ARIMA(maxiter=maxiter, order=order)
            mlflow.log_params({"p": p,"d": d,"q": q,"maxiter":maxiter,"n_periods": n_periods, "prediction_start_date":split.index.to_list()[0].strftime('%d-%m-%Y')})
            for t in range(len(split)):
                model.fit(train_recursive)
                forecast = model.predict(n_periods=n_periods)
                predictions.append(forecast.tolist())
                train_recursive.append(split.iloc[t])
            df = pd.DataFrame(predictions, columns=[f"t{i+1}" for i in range(len(predictions[0]))])
            mse_list = []
            rmse_list = []
            rmspe_list = []
            mae_list = []
            mape_list = []
            accuracy_list  = []
            precision_list = []
            for i in range(len(df.columns)):
                df[f't{i+1}_prc_dir'] = df[f't{i+1}'].diff().apply(lambda x: 1 if x > 0 else -1)
                mse = mean_squared_error(val[i:-n_periods+i], df[f't{i+1}'])
                rmse = root_mean_squared_error(val[i:-n_periods+i],  df[f't{i+1}'])
                rmspe = root_mean_squared_percentage_error(val[i:-n_periods+i],  df[f't{i+1}'])
                mae = mean_absolute_error(val[i:-n_periods+i],  df[f't{i+1}'])
                mape = mean_absolute_percentage_error(val[i:-n_periods+i],  df[f't{i+1}'])
                accuracy = accuracy_score(val[i:-n_periods+i].diff().apply(lambda x: 1 if x > 0 else -1).dropna(), df[f't{i+1}_prc_dir'].dropna())
                precision = accuracy_score(val[i:-n_periods+i].diff().apply(lambda x: 1 if x > 0 else -1).dropna(), df[f't{i+1}_prc_dir'].dropna())
                mse_list.append(mse)
                rmse_list.append(rmse)
                rmspe_list.append(rmspe)
                mae_list.append(mae)
                mape_list.append(mape)
                accuracy_list.append(accuracy)
                precision_list.append(precision)
                avg_mse = sum(mse_list)/len(mse_list)
                avg_rmse = sum(rmse_list)/len(rmse_list)
                avg_mae = sum(mae_list)/len(mae_list)
                avg_mape = sum(mape_list)/len(mape_list)
                avg_accuracy = sum(accuracy_list)/len(accuracy_list)
                avg_precision = sum(precision_list)/len(precision_list)
                mlflow.log_metrics({
                    "mse_daily":mse_list[i], 
                    "rmse_daily":rmse_list[i],
                    "rmspe_daily":rmspe_list[i], 
                    "mae_daily":mae_list[i], 
                    "mape_daily":mape_list[i], 
                    "accuracy_daily":accuracy_list[i],
                    "precision_daily":precision_list[i],
                    "avg_mse": avg_mse,
                    "avg_rmse": avg_rmse,
                    "avg_mae": avg_mae,
                    "avg_mape": avg_mape,
                    "avg_accuracy":avg_accuracy,
                    "avg_precision":avg_precision
                }, step=(i+1))
        mlflow.end_run()
        results_file = f"{run_name}_{split_name}"
        path = str(os.path.join(root_dir,"mlruns",model_type))
        df.to_csv(str(os.path.join(path,results_file)), index=False)
        print(f"Completed order: {order}")

Completed order: (1, 1, 1)
Completed order: (2, 1, 1)
Completed order: (3, 1, 1)
Completed order: (4, 1, 1)
Completed order: (5, 1, 1)
Completed order: (6, 1, 1)
Completed order: (7, 1, 1)


In [7]:
mlflow.set_tracking_uri(f"sqlite:///{root_dir}/mlruns/mlruns.db")
mlflow.set_experiment(experiment_name)

predictions = []
train_recursive = list(train)
order = (test_p, d, test_q)
run_name = f"order_({test_p},{d},{test_q})"
with mlflow.start_run(run_name=run_name):
    model = ARIMA(maxiter=maxiter, order=order)
    mlflow.log_params({"p": test_p,"d": d,"q": test_q,"maxiter":maxiter,"n_periods": n_periods, "prediction_start_date":split.index.to_list()[0].strftime('%d-%m-%Y')})
    for t in range(len(split)):
        model.fit(train_recursive)
        forecast = model.predict(n_periods=n_periods)
        predictions.append(forecast.tolist())
        train_recursive.append(split.iloc[t])
    df = pd.DataFrame(predictions, columns=[f"t{i+1}" for i in range(len(predictions[0]))])
    mse_list = []
    rmse_list = []
    rmspe_list = []
    mae_list = []
    mape_list = []
    accuracy_list  = []
    precision_list = []
    for i in range(len(df.columns)):
        df[f't{i+1}_prc_dir'] = df[f't{i+1}'].diff().apply(lambda x: 1 if x > 0 else -1)
        mse = mean_squared_error(test[i:-n_periods+i], df[f't{i+1}'])
        rmse = root_mean_squared_error(test[i:-n_periods+i],  df[f't{i+1}'])
        rmspe = root_mean_squared_percentage_error(test[i:-n_periods+i],  df[f't{i+1}'])
        mae = mean_absolute_error(test[i:-n_periods+i],  df[f't{i+1}'])
        mape = mean_absolute_percentage_error(test[i:-n_periods+i],  df[f't{i+1}'])
        accuracy = accuracy_score(test[i:-n_periods+i].diff().apply(lambda x: 1 if x > 0 else -1).dropna(), df[f't{i+1}_prc_dir'].dropna())
        precision = accuracy_score(test[i:-n_periods+i].diff().apply(lambda x: 1 if x > 0 else -1).dropna(), df[f't{i+1}_prc_dir'].dropna())
        mse_list.append(mse)
        rmse_list.append(rmse)
        rmspe_list.append(rmspe)
        mae_list.append(mae)
        mape_list.append(mape)
        accuracy_list.append(accuracy)
        precision_list.append(precision)
        avg_mse = sum(mse_list)/len(mse_list)
        avg_rmse = sum(rmse_list)/len(rmse_list)
        avg_mae = sum(mae_list)/len(mae_list)
        avg_mape = sum(mape_list)/len(mape_list)
        avg_accuracy = sum(accuracy_list)/len(accuracy_list)
        avg_precision = sum(precision_list)/len(precision_list)
        mlflow.log_metrics({
            "mse_daily":mse_list[i], 
            "rmse_daily":rmse_list[i],
            "rmspe_daily":rmspe_list[i], 
            "mae_daily":mae_list[i], 
            "mape_daily":mape_list[i], 
            "accuracy_daily":accuracy_list[i],
            "precision_daily":precision_list[i],
            "avg_mse": avg_mse,
            "avg_rmse": avg_rmse,
            "avg_mae": avg_mae,
            "avg_mape": avg_mape,
            "avg_accuracy":avg_accuracy,
            "avg_precision":avg_precision
        }, step=(i+1))
mlflow.end_run()
results_file = f"{run_name}_{split_name}"
path = str(os.path.join(root_dir,"mlruns",model_type))
df.to_csv(str(os.path.join(path,results_file)), index=False)
print(f"Completed order: {order}")

Completed order: (1, 1, 1)


In [19]:
from mlflow.tracking import MlflowClient
import pandas as pd

# Config
tracking_uri = f"sqlite:///{root_dir}/mlruns/mlruns.db"
experiment_name = experiment_name
metric_name = "rmspe_daily"
avg_metric_name = "avg_rmspe"

# Init MLflow client
client = MlflowClient(tracking_uri=tracking_uri)

# Get experiment ID from name
experiment = client.get_experiment_by_name(experiment_name)
if experiment is None:
    raise ValueError(f"Experiment '{experiment_name}' not found.")
experiment_id = experiment.experiment_id

# Search all runs in experiment
runs = client.search_runs([experiment_id])

# Collect metric history from all runs
records = []
for run in runs:
    run_id = run.info.run_id
    history = client.get_metric_history(run_id, metric_name)
    for m in history:
        records.append({
            "run_id": run_id,
            "step": m.step,
            "value": m.value
        })

# Create DataFrame
df = pd.DataFrame(records)

# Pivot so each step is a column
df_pivot = df.pivot(index="run_id", columns="step", values="value")
df_pivot = df_pivot.apply(pd.to_numeric, errors="coerce")

# Compute average per run
df_pivot["average"] = df_pivot.mean(axis=1, skipna=True)

# Log average metric back to MLflow for each run
for run_id, avg_value in df_pivot["average"].items():
    client.log_metric(run_id, avg_metric_name, float(avg_value))

print("Averages logged back to MLflow successfully.")

Averages logged back to MLflow successfully.


In [25]:
import mlflow
import pandas as pd

# Experiment name
tracking_uri = f"sqlite:///{root_dir}/mlruns/mlruns.db"
experiment_name = experiment_name

# Metric names
metric1 = "avg_rmspe"
metric2 = "avg_precision"

# Create MLflow client
client = MlflowClient(tracking_uri=tracking_uri)

# Get experiment ID from name
experiment = client.get_experiment_by_name(experiment_name)
if experiment is None:
    raise ValueError(f"Experiment '{experiment_name}' not found.")
experiment_id = experiment.experiment_id

# Search all runs in the experiment
runs = client.search_runs(
    experiment_ids=[experiment_id],
    filter_string="",
    run_view_type=mlflow.entities.ViewType.ACTIVE_ONLY,
    max_results=5000
)

data = []
for run in runs:
    m1 = run.data.metrics.get(metric1)
    m2 = run.data.metrics.get(metric2)*100

    ratio = m1 / m2 if m1 is not None and m2 not in (None, 0) else None

    data.append({
        "run_id": run.info.run_id,
        metric1: m1,
        metric2: m2,
        "ratio": ratio
    })

# Convert to DataFrame
df = pd.DataFrame(data)

print(df)

                             run_id  avg_rmspe  avg_precision     ratio
0  b56091755a8c40ac9917c461062d1e1e   4.182459      49.567100  0.084380
1  c6c90a0a459f4382b6457d500a5541df   4.259540      48.701299  0.087463
2  fb728d8a61284625a2a21617ea9d5ebe   4.224460      49.025974  0.086168
3  eb4bb50df68a4b99b1806e89adf4e818   4.167074      49.242424  0.084624
4  11c1d25adb8a49629d31c64c7ea2994b   4.198147      49.025974  0.085631
5  4de8aa3d89f94f71a016bd333e0818ab   4.164318      49.783550  0.083648
6  391f067cf4c549369e4880d45fcf7020   4.154483      50.324675  0.082554
